<a href="https://colab.research.google.com/github/a01739638-droid/Extraccion-de-Datos/blob/main/Citas_Digitales_General.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import pandas as pd

In [21]:
df = pd.read_excel("Citas_Digital.xlsx")

df = df.iloc[0:674].reset_index(drop=True)

columnas = [0, 1, 2, 4, 5, 6, 7, 8, 9]
df = df.iloc[:, columnas]

def limpiar_potencial(valor):

    if pd.isna(valor):
        return None
    if isinstance(valor, (int, float)):
        return float(valor)

    texto = str(valor).strip()
    # Deja solo dígitos y el punto decimal
    texto_limpio = "".join(c for c in texto if c.isdigit() or c == ".")

    if texto_limpio == "":
        # Ej. "Ventas " no es un valor numérico -> se marca como nulo
        return None

    numero = float(texto_limpio)
    # Si el valor es mayor a 1, se asume que venía en formato "porcentaje"
    # mal escrito (ej. "80%%" -> 80 -> 0.80)
    if numero > 1:
        numero = numero / 100
    return numero


df["Potencial de compra"] = df["Potencial de compra"].apply(limpiar_potencial).astype(float)

# --- Paso 4: Obtener el CSV general ---
df.to_csv("Citas_Digital_General.csv", index=False, encoding="utf-8-sig")

print("Filas:", df.shape[0], "| Columnas:", df.shape[1])
print(df.dtypes)
print(df.head())

Filas: 674 | Columnas: 9
BDC                            object
Fecha                  datetime64[ns]
Nombre Cliente                 object
Estatus de Lead                object
Potencial de compra           float64
PDM                           float64
SDC                           float64
Venta                         float64
Asesor Asignado                object
dtype: object
    BDC  Fecha Nombre Cliente  Estatus de Lead  Potencial de compra  PDM  SDC  \
0  Saul    NaT      Guillermo      Finalizado                  0.90  1.0  0.0   
1  Saul    NaT        Roberto      Finalizado                  0.40  0.0  0.0   
2  Saul    NaT         Irving      Finalizado                  0.85  0.0  0.0   
3  Saul    NaT        Oswaldo           Venta                  0.90  0.0  0.0   
4  Saul    NaT        Rodrigo      Finalizado                  0.90  1.0  0.0   

   Venta Asesor Asignado  
0    0.0           Nancy  
1    0.0         Victor   
2    0.0           Alan   
3    1.0       Mauricio 

In [22]:
import pandas as pd
import os

# --- Cargar el CSV general ---
df = pd.read_csv("Citas_Digital_General.csv", encoding="utf-8-sig")

# Convertir la columna Fecha a tipo fecha (los valores vacíos quedan como NaT)
df["Fecha"] = pd.to_datetime(df["Fecha"], errors="coerce")

# --- Carpeta de salida (opcional, para tener los 16 csv organizados) ---
carpeta_salida = "csv_por_mes"
os.makedirs(carpeta_salida, exist_ok=True)

# Nombres de mes en español
meses_es = {
    1: "Enero", 2: "Febrero", 3: "Marzo", 4: "Abril",
    5: "Mayo", 6: "Junio", 7: "Julio", 8: "Agosto",
    9: "Septiembre", 10: "Octubre", 11: "Noviembre", 12: "Diciembre",
}

# --- Generar la lista de meses: Abril 2025 -> Julio 2026 (16 meses) ---
periodos = pd.period_range(start="2025-04", end="2026-07", freq="M")

resumen = []

for periodo in periodos:
    anio = periodo.year
    mes_num = periodo.month
    nombre_mes = meses_es[mes_num]

    # Filtro por filas: solo las citas de ese año y mes
    filtro = (df["Fecha"].dt.year == anio) & (df["Fecha"].dt.month == mes_num)
    df_mes = df[filtro]

    nombre_archivo = f"{nombre_mes}_{anio}.csv"
    ruta = os.path.join(carpeta_salida, nombre_archivo)
    df_mes.to_csv(ruta, index=False, encoding="utf-8-sig")

    resumen.append((nombre_archivo, df_mes.shape[0]))

# --- Resumen de filas por archivo generado ---
print(f"{'Archivo':<25} Filas")
print("-" * 35)
for nombre, filas in resumen:
    print(f"{nombre:<25} {filas}")

print(f"\nTotal de archivos generados: {len(resumen)}")

Archivo                   Filas
-----------------------------------
Abril_2025.csv            0
Mayo_2025.csv             0
Junio_2025.csv            0
Julio_2025.csv            1
Agosto_2025.csv           0
Septiembre_2025.csv       0
Octubre_2025.csv          0
Noviembre_2025.csv        0
Diciembre_2025.csv        0
Enero_2026.csv            0
Febrero_2026.csv          0
Marzo_2026.csv            0
Abril_2026.csv            0
Mayo_2026.csv             0
Junio_2026.csv            44
Julio_2026.csv            28

Total de archivos generados: 16


In [23]:
import shutil
shutil.make_archive("csv_por_mes", "zip", "csv_por_mes")

from google.colab import files
files.download("csv_por_mes.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>